In [1]:
# Download the Cornell Movie-Dialogs Corpus (two text files) into ./datasets
import os, time, urllib.error, urllib.request, zipfile

URL = "https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip"
ZIP_PATH = "cornell.zip"


def download(url: str, path: str, attempts: int = 8) -> None:
    """Download with resume: this server often drops the connection mid-file."""
    for attempt in range(attempts):
        have = os.path.getsize(path) if os.path.exists(path) else 0
        request = urllib.request.Request(url, headers={"Range": f"bytes={have}-"})
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                # 206: the server resumed from `have`; 200: it is sending the whole file again
                resumed = response.status == 206
                total = int(response.headers["Content-Length"]) + (
                    have if resumed else 0
                )
                with open(path, "ab" if resumed else "wb") as f:
                    while chunk := response.read(1 << 20):
                        f.write(chunk)
            if os.path.getsize(path) == total:
                return
        except urllib.error.HTTPError as err:
            # 416: nothing left to send, the file is already complete
            if err.code == 416 and zipfile.is_zipfile(path):
                return
            print(f"attempt {attempt + 1}: {err}")
        except (urllib.error.URLError, ConnectionError, TimeoutError) as err:
            print(f"attempt {attempt + 1}: {err}; resuming")
        time.sleep(2)
    raise RuntimeError(f"Could not download {url} after {attempts} attempts")


if not os.path.exists("datasets/movie_lines.txt"):
    download(URL, ZIP_PATH)
    os.makedirs("datasets", exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for name in ["movie_conversations.txt", "movie_lines.txt"]:
            member = next(
                m
                for m in archive.namelist()
                if m.endswith(name) and "__MACOSX" not in m
            )
            with archive.open(member) as src, open(f"datasets/{name}", "wb") as dst:
                dst.write(src.read())
    os.remove(ZIP_PATH)
print(os.listdir("datasets"))

['movie_lines.txt', 'movie_conversations.txt']


In [2]:
# Import standard libraries and PyTorch modules for building the Transformer
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.utils.data
import ast
import math
import time
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {DEVICE}")

Using device: mps


# 1) Data Processing

`movie_conversations.txt` contains list of the conversation IDs and `movie_lines.text` contains the text of assoicated with each conversation ID.


In [3]:
# Parameter and Set Up
max_len = 25


def remove_punc(string):
    punctuations = """!()-[]{};:'"\,<>./?@#$%^&*_~"""
    no_punct = ""
    for char in string:
        if char not in punctuations:
            # space is also a character
            no_punct = no_punct + char
    return no_punct.lower()


corpus_movie_conv = "./datasets/movie_conversations.txt"
corpus_movie_lines = "./datasets/movie_lines.txt"
with open(corpus_movie_conv, "r", encoding="iso-8859-1") as c:
    conv = c.readlines()
with open(corpus_movie_lines, "r", encoding="iso-8859-1") as l:
    lines = l.readlines()

# Map line IDs to their corresponding text content
lines_dic = {}
for line in lines:
    objects = line.split(" +++$+++ ")
    lines_dic[objects[0]] = objects[-1]

# Parse conversation sequences and generate question-answer pairs
# Output: List of [question_tokens, answer_tokens], each truncated to max_len
pairs = []
for con in conv:
    ids = ast.literal_eval(con.split(" +++$+++ ")[-1])
    for i in range(len(ids)):
        qa_pairs = []

        if i == len(ids) - 1:
            break

        # Clean and tokenize sentence strings
        first = remove_punc(lines_dic[ids[i]].strip())
        second = remove_punc(lines_dic[ids[i + 1]].strip())

        # Truncate sequences to the defined max_len
        # Output: (num_tokens,) limited by max_len
        qa_pairs.append(first.split()[:max_len])
        qa_pairs.append(second.split()[:max_len])
        pairs.append(qa_pairs)

# Print a sample pair to verify processing
print(pairs[20])

[['i', 'really', 'really', 'really', 'wanna', 'go', 'but', 'i', 'cant', 'not', 'unless', 'my', 'sister', 'goes'], ['im', 'workin', 'on', 'it', 'but', 'she', 'doesnt', 'seem', 'to', 'be', 'goin', 'for', 'him']]


In [4]:
# Count frequency of each word across all conversation pairs
min_word_freq = 5
word_freq = Counter()

for pair in pairs:
    word_freq.update(pair[0])
    word_freq.update(pair[1])

# vocab_size = number of words above threshold + 4 special tokens
words = [w for w in word_freq.keys() if word_freq[w] > min_word_freq]
word_map = {k: v + 1 for v, k in enumerate(words)}
word_map["<unk>"] = len(word_map) + 1
word_map["<start>"] = len(word_map) + 1
word_map["<end>"] = len(word_map) + 1
word_map["<pad>"] = 0

print("Total words are: {}".format(len(word_map)))


# Encode a question (no start/end tokens) into a padded integer sequence
# (num_tokens,) → (max_len,)
def encode_question(words, word_map):
    enc_c = [word_map.get(word, word_map["<unk>"]) for word in words] + [
        word_map["<pad>"]
    ] * (max_len - len(words))
    return enc_c


# Encode a reply (with <start> and <end> tokens) into a padded integer sequence
# (num_tokens,) → (max_len + 2,)
def encode_reply(words, word_map):
    enc_c = (
        [word_map["<start>"]]
        + [word_map.get(word, word_map["<unk>"]) for word in words]
        + [word_map["<end>"]]
        + [word_map["<pad>"]] * (max_len - len(words))
    )
    return enc_c


# Encode all conversation pairs into numerical format
# List of string token lists → List of [[max_len], [max_len + 2]]
pairs_encoded = []
for pair in pairs:
    qus = encode_question(pair[0], word_map)
    ans = encode_reply(pair[1], word_map)
    pairs_encoded.append([qus, ans])

Total words are: 18243

In [5]:
# Define the custom Dataset class for movie conversation pairs
class Dataset(torch.utils.data.Dataset):

    def __init__(self, pairs):
        self.pairs = pairs
        self.dataset_size = len(self.pairs)

    # Retrieve a single encoded question/reply pair as LongTensors
    # Output: question (max_len,), reply (max_len + 2,)
    def __getitem__(self, i):
        question = torch.LongTensor(self.pairs[i][0])
        reply = torch.LongTensor(self.pairs[i][1])
        return question, reply

    def __len__(self):
        return self.dataset_size


# Initialize the DataLoader to manage batches of encoded pairs
# Output per batch: question (batch_num, max_len), reply (batch_num, max_len + 2)
train_loader = DataLoader(
    Dataset(pairs_encoded), batch_size=32, shuffle=True, pin_memory=True
)

# Extract a sample batch to verify tensor shapes
question, reply = next(iter(train_loader))
print("Question: ", question.size())
print("Answer: ", reply.size())

Question:  torch.Size([32, 25])
Answer:  torch.Size([32, 27])


# 2) Masking

1. Mask all the pad tokens (value `0`) in the batch to ensure the model does not treat padding as input.

2. **Look-ahead Mask** to mask the future tokens in a sequence.
We also mask out pad tokens. i.e. To predict the third word, only the first and second word will be used

In [6]:
# Function to generate padding and look-ahead masks for the encoder and decoder
def create_masks(question, reply_input, reply_target):

    # Helper: lower-triangular mask so position t only sees positions <= t
    # seq_len (scalar) → (1, seq_len, seq_len)
    def subsequent_mask(size, DEVICE):
        return torch.tril(
            torch.ones(size, size, dtype=torch.bool, device=DEVICE)
        ).unsqueeze(0)

    # Create a padding mask for the encoder input (True where token != pad)
    # (batch_num, max_len) → (batch_num, 1, 1, max_len)
    question_mask = (question != 0).unsqueeze(1).unsqueeze(1)

    # Create a padding mask for the decoder input
    # (batch_num, output_len) → (batch_num, 1, output_len)
    reply_input_mask = (reply_input != 0).unsqueeze(1)

    # Combine padding mask with look-ahead mask for decoder self-attention
    # Input: (batch_num, 1, output_len) & (1, output_len, output_len)
    # Output: (batch_num, output_len, output_len)
    reply_input_mask = reply_input_mask & subsequent_mask(
        reply_input.size(-1), reply_input.device
    )
    # Add head dimension: (batch_num, 1, output_len, output_len)
    reply_input_mask = reply_input_mask.unsqueeze(1)

    # Create a mask for the decoder target sequence (True where token != pad)
    # (batch_num, output_len)
    reply_target_mask = reply_target != 0

    return question_mask, reply_input_mask, reply_target_mask


# Right-shift reply to create teacher-forced decoder input and target
# (batch_num, output_len + 1)
reply_input = reply[:, :-1]
reply_target = reply[:, 1:]
print("Reply Target Size: ", reply_target.size())

# Generate all required masks for training
# question_mask:     (batch_num, 1, 1, max_len)
# reply_input_mask:  (batch_num, 1, output_len + 1, output_len + 1)
# reply_target_mask: (batch_num, output_len + 1)
question_mask, reply_input_mask, reply_target_mask = create_masks(
    question, reply_input, reply_target
)
print("question_mask Size: ", question_mask.size())
print("reply_input_mask Size: ", reply_input_mask.size())
print("reply_target_mask Size: ", reply_target_mask.size())

Reply Target Size:  torch.Size([32, 26])
question_mask Size:  torch.Size([32, 1, 1, 25])
reply_input_mask Size:  torch.Size([32, 1, 26, 26])
reply_target_mask Size:  torch.Size([32, 26])


# 3) Embedding

## 3.1 Positional Embedding
The positional encoding vector is added to the embedding vector. Embeddings represent a token in a d-dimensional space where tokens with similar meaning will be closer to each other. But the embeddings do not encode the relative position of words in a sentence. So after adding the positional encoding, words will be closer to each other based on the *similarity of their meaning and their position in the sentence*, in the d-dimensional space.

<div align="center">
  <img src="https://substack-post-media.s3.amazonaws.com/public/images/5f05e827-c1c0-4bc9-9640-cbb43b8c47a6_926x826.png" width="500" alt="Sinusoidal positional encoding: sin on even dimensions 2i, cos on odd dimensions 2i+1">
</div>



$$\Large{PE_{(pos, 2i)} = sin(pos / 10000^{2i / d_{model}})} $$
$$\Large{PE_{(pos, 2i+1)} = cos(pos / 10000^{2i / d_{model}})} $$

In [7]:
# Token embeddings scaled by sqrt(embed_dim) plus fixed sinusoidal position encodings
class Embeddings(nn.Module):
    """
    Implements embeddings of the words and adds their positional encodings.
    """

    def __init__(self, vocab_size, embed_dim, max_len=50):
        super(Embeddings, self).__init__()
        self.embed_dim = embed_dim

        # Token embedding lookup table
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        self.embed = nn.Embedding(vocab_size, embed_dim)

        # Buffer, not parameter: moves with .to(DEVICE) but is never trained
        # (1, max_len, embed_dim)
        self.register_buffer("pe", self.create_positional_encoding(max_len, embed_dim))
        self.dropout = nn.Dropout(0.1)

    @staticmethod
    def create_positional_encoding(max_len, embed_dim):
        # (max_len, 1)
        pos = torch.arange(max_len).unsqueeze(1)

        # 10000^(2i / embed_dim) for every dimension pair i
        # (embed_dim // 2,)
        div = 10000 ** (torch.arange(0, embed_dim, 2) / embed_dim)

        # sin on even dims 2i, cos on odd dims 2i+1, same frequency within a pair
        # (max_len, embed_dim)
        pe = torch.zeros(max_len, embed_dim)
        pe[:, 0::2] = torch.sin(pos / div)
        pe[:, 1::2] = torch.cos(pos / div)

        # (max_len, embed_dim) → (1, max_len, embed_dim)
        return pe.unsqueeze(0)

    def forward(self, tokens):
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        embedding = self.embed(tokens) * math.sqrt(self.embed_dim)

        # Add sinusoidal positional encoding
        # (batch_num, seq_len, embed_dim)
        embedding = embedding + self.pe[:, : tokens.size(1)]
        return self.dropout(embedding)


# Instantiate the embedding model
embed_model = Embeddings(len(word_map), 512)

# Test the embedding forward pass
# (batch_num, max_len) → (batch_num, max_len, embed_dim)
result = embed_model(question)
print(result.size())

torch.Size([32, 25, 512])


# 4) Transformers

## 4.1 Scaled dot product Attention

The scaled dot-product attention function used by the transformer takes three inputs: Q (query), K (key), V (value). The equation used to calculate the attention weights is:

$$\Large{Attention(Q, K, V) = softmax_k(\frac{QK^T}{\sqrt{d_k}} + M) V} $$

As the softmax normalization is done on the `key`, its values decide the amount of importance given to the `query`.

The output represents the multiplication of the attention weights and the `value` vector. This ensures that the words we want to focus on are kept as is and the irrelevant words are flushed out.

The dot-product attention is scaled by a factor of square root of the depth. This is done because for large values of depth, the dot product grows large in magnitude pushing the softmax function where it has small gradients resulting in a very hard softmax.

For example, consider that `query` and `key` have a mean of 0 and variance of 1. Their matrix multiplication will have a mean of 0 and variance of `dk`. Hence, *square root of `dk`* is used for scaling (and not any other number) because the matmul of `query` and `key` should have a mean of 0 and variance of 1, so that we get a gentler softmax.

Masking is needed to prevent the attention mechanism of a transformer from “cheating” in the decoder or peeking to the future. The mask is multiplied with *-1e9 (close to negative infinity).* This is done because the mask is summed with the scaled matrix multiplication of `query` and `key` and is applied immediately before a softmax. The goal is to zero out these cells, and large negative inputs to softmax are near zero in the output.

## 4.2 Multi-head attention

<div align="center">
<img src="https://www.tensorflow.org/images/tutorials/transformer/multi_head_attention.png" width="300" alt="multi-head attention">
</div>


Multi-head attention consists of four parts:
* Linear layers and split into heads.
* Scaled dot-product attention.
* Concatenation of heads.
* Final linear layer.

Each multi-head attention block gets three inputs; Q (query), K (key), V (value). These are put through linear (Dense) layers and split up into multiple heads.

Scaled dot-product attention is applied to every head at once (the head axis is a batch dimension). An appropriate mask must be used in the attention step. The attention output for each head is then concatenated (`permute` + `view`) and put through a final `nn.Linear` layer.

Instead of one single attention head, `query`, `key`, and `value` are split into multiple heads because it allows the model to jointly attend to information at different positions from different representational spaces. After the split each head has a reduced dimensionality, so the total computation cost is the same as a single head attention with full dimensionality.

In [8]:
class MultiHeadAttention(nn.Module):

    def __init__(self, heads, embed_dim, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert embed_dim % heads == 0
        # head dimension = model_dim / num_heads
        self.d_k = embed_dim // heads
        self.heads = heads
        self.dropout = nn.Dropout(dropout)

        # Linear projections for Q, K, V and output
        # Each: (batch_num, seq_len, model_dim)
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, mask):
        # Project inputs to query, key, and value representations
        # (batch_num, seq_len, model_dim)
        query = self.query(query)
        key = self.key(key)
        value = self.value(value)

        # Reshape and transpose to split into num_heads
        # (batch_num, seq_len, model_dim) → (batch_num, num_heads, seq_len, d_k)
        query = query.view(query.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        key = key.view(key.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        value = value.view(value.shape[0], -1, self.heads, self.d_k).permute(0, 2, 1, 3)

        # Scaled dot-product attention scores
        # Input: (batch_num, num_heads, query_len, d_k) x (batch_num, num_heads, d_k, key_len)
        # Output: (batch_num, num_heads, query_len, key_len)
        scores = torch.matmul(query, key.permute(0, 1, 3, 2)) / math.sqrt(
            query.size(-1)
        )

        # Mask out padding/future positions with -1e9 before softmax
        # Input/Output: (batch_num, num_heads, query_len, key_len)
        scores = scores.masked_fill(mask == 0, -1e9)

        # Convert scores to attention weights via softmax
        # Output: (batch_num, num_heads, query_len, key_len)
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Weighted sum over value vectors
        # Input: (batch_num, num_heads, query_len, key_len) x (batch_num, num_heads, key_len, d_k)
        # Output: (batch_num, num_heads, query_len, d_k)
        context = torch.matmul(weights, value)

        # Merge heads back: transpose then reshape
        # (batch_num, num_heads, query_len, d_k) → (batch_num, query_len, model_dim)
        context = (
            context.permute(0, 2, 1, 3)
            .contiguous()
            .view(context.shape[0], -1, self.heads * self.d_k)
        )

        # Final linear projection to model_dim
        # (batch_num, query_len, model_dim)
        return self.output_linear(context)

## 4.3 FeedForward Layer

Network with one hidden layer that applies a non-linear activation function (GELU) to the output of the first linear layer

In [9]:
# Position-wise Feed-Forward Network: two linear layers with GELU activation and dropout
class FeedForward(nn.Module):
    def __init__(self, embed_dim, middle_dim=2048):
        super(FeedForward, self).__init__()

        self.fc1 = nn.Linear(embed_dim, middle_dim)
        self.fc2 = nn.Linear(middle_dim, embed_dim)
        self.dropout = nn.Dropout(0.1)
        self.activation = torch.nn.GELU()

    def forward(self, x):
        # First linear layer with GELU activation
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, middle_dim)
        out = self.activation(self.fc1(x))

        # Dropout then second linear layer to project back to model_dim
        # (batch_num, seq_len, middle_dim) → (batch_num, seq_len, model_dim)
        out = self.fc2(self.dropout(out))

        return out

## 4.4 Encoder Layer

<img src="https://naokishibuya.github.io/blog/2021-12-13-transformers-encoder-decoder/images/encoder-layer-norm.png" width='500'>

Each encoder layer consists of sublayers:

1. Multi-head attention (with padding mask)
2. 2 dense layers followed by dropout

Each of these sublayers has a residual connection around it followed by a layer normalization. Residual connections help in avoiding the vanishing gradient problem in deep networks.

The output of each sublayer is `LayerNorm(x + Sublayer(x))`. The normalization is done on the `embed_dim` (last) axis.

In [10]:
# Encoder Layer: self-attention + feed-forward with residual connections and layer normalization
class EncoderLayer(nn.Module):

    def __init__(self, embed_dim, heads):
        super(EncoderLayer, self).__init__()
        # One LayerNorm per sublayer: each learns its own gain and bias
        self.attn_norm = nn.LayerNorm(embed_dim)
        self.ff_norm = nn.LayerNorm(embed_dim)
        self.self_multihead = MultiHeadAttention(heads, embed_dim)
        self.feed_forward = FeedForward(embed_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, embeddings, mask):
        # Self-attention over encoder inputs
        # (batch_num, seq_len, model_dim)
        interacted = self.dropout(
            self.self_multihead(embeddings, embeddings, embeddings, mask)
        )

        # First residual connection + layer normalization
        # (batch_num, seq_len, model_dim)
        interacted = self.attn_norm(interacted + embeddings)

        # Position-wise feed-forward network
        # (batch_num, seq_len, model_dim)
        feed_forward_out = self.dropout(self.feed_forward(interacted))

        # Second residual connection + layer normalization
        # (batch_num, seq_len, model_dim)
        encoded = self.ff_norm(feed_forward_out + interacted)

        return encoded

## 4.5 Decoder Layer

Each decoder layer consists of sublayers:

1.   Masked multi-head attention (with look ahead mask and padding mask)
2.   Multi-head attention (with padding mask). `value` and `key` receive the *encoder output* as inputs. `query` receives the *output from the masked multi-head attention sublayer.*
3.   2 dense layers followed by dropout

<div align="center">
<img src="https://naokishibuya.github.io/blog/2021-12-13-transformers-encoder-decoder/images/decoder-source-target-2.png" width='400'>
</div>

Each of these sublayers has a residual connection around it followed by a layer normalization. The output of each sublayer is `LayerNorm(x + Sublayer(x))`. The normalization is done on the `embed_dim` (last) axis.

As `query` receives the output from decoder's first attention block, and `key` receives the encoder output, the attention weights represent the importance given to the decoder's input based on the encoder's output. In other words, the decoder predicts the next word by looking at the encoder output and self-attending to its own output. See the demonstration above in the scaled dot product attention section.

In [11]:
class DecoderLayer(nn.Module):

    def __init__(self, embed_dim, heads):
        super(DecoderLayer, self).__init__()
        # One LayerNorm per sublayer
        self.self_attn_norm = nn.LayerNorm(embed_dim)
        self.cross_attn_norm = nn.LayerNorm(embed_dim)
        self.ff_norm = nn.LayerNorm(embed_dim)
        self.self_multihead = MultiHeadAttention(heads, embed_dim)
        self.src_multihead = MultiHeadAttention(heads, embed_dim)
        self.feed_forward = FeedForward(embed_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, embeddings, encoded, src_mask, target_mask):

        # Masked self-attention over decoder inputs (prevents attending to future tokens)
        # (batch_num, output_len, model_dim)
        query = self.dropout(
            self.self_multihead(embeddings, embeddings, embeddings, target_mask)
        )

        # First residual connection + layer normalization
        # Input/Output: (batch_num, output_len, model_dim)
        query = self.self_attn_norm(query + embeddings)

        # Cross-attention: decoder queries attend to encoder keys/values
        # query: (batch_num, output_len, model_dim), encoded: (batch_num, input_len, model_dim)
        # Output: (batch_num, output_len, model_dim)
        interacted = self.dropout(self.src_multihead(query, encoded, encoded, src_mask))

        # Second residual connection + layer normalization
        # (batch_num, output_len, model_dim)
        interacted = self.cross_attn_norm(interacted + query)

        # Position-wise feed-forward network
        # (batch_num, output_len, model_dim) → (batch_num, output_len, model_dim)
        feed_forward_out = self.dropout(self.feed_forward(interacted))

        # Third residual connection + layer normalization
        # (batch_num, output_len, model_dim)
        decoded = self.ff_norm(feed_forward_out + interacted)

        return decoded

## 4.6 Transformer

Transformer consists of the encoder, decoder and a final linear layer. The output of the decoder is the input to the linear layer and its output is returned.

<img src='https://lilianweng.github.io/posts/2020-04-07-the-transformer-family/transformer.png' width='500'>

<img src='https://1.bp.blogspot.com/-AVGK0ApREtk/WaiAuzddKVI/AAAAAAAAB_A/WPV5ropBU-cxrcMpqJBFHg73K9NX4vywwCLcBGAs/s1600/image2.png' width='500'>

In [12]:
class Transformer(nn.Module):

    def __init__(self, embed_dim, heads, num_layers, word_map):
        super(Transformer, self).__init__()

        self.embed_dim = embed_dim
        self.num_layers = num_layers
        self.vocab_size = len(word_map)
        # Shared embedding table: questions and replies use the same vocabulary
        self.embed = Embeddings(self.vocab_size, embed_dim)
        # num_layers independent layers on each side
        self.encoder = nn.ModuleList(
            [EncoderLayer(embed_dim, heads) for _ in range(num_layers)]
        )
        self.decoder = nn.ModuleList(
            [DecoderLayer(embed_dim, heads) for _ in range(num_layers)]
        )
        # Final linear projection to vocabulary logits
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, vocab_size)
        self.logit = nn.Linear(embed_dim, self.vocab_size)

    def encode(self, src_words, src_mask):
        # (batch_num, input_len) → (batch_num, input_len, embed_dim)
        x = self.embed(src_words)
        for layer in self.encoder:
            # (batch_num, input_len, embed_dim)
            x = layer(x, src_mask)
        return x

    def decode(self, tgt_words, target_mask, src_embeddings, src_mask):
        # Input tgt: (batch_num, output_len), src: (batch_num, input_len, embed_dim)
        # Output: (batch_num, output_len, embed_dim)
        x = self.embed(tgt_words)
        for layer in self.decoder:
            x = layer(x, src_embeddings, src_mask, target_mask)
        return x

    def forward(self, src_words, src_mask, target_words, target_mask):
        # Encode source sequence
        # (batch_num, input_len) → (batch_num, input_len, embed_dim)
        encoded = self.encode(src_words, src_mask)

        # Decode target sequence conditioned on encoder output
        # Input tgt: (batch_num, output_len), src: (batch_num, input_len, embed_dim)
        # Output: (batch_num, output_len, embed_dim)
        decoded = self.decode(target_words, target_mask, encoded, src_mask)

        # Project to vocabulary and apply log-softmax
        # (batch_num, output_len, embed_dim) → (batch_num, output_len, vocab_size)
        out = F.log_softmax(self.logit(decoded), dim=2)
        return out

# 5) Optimizer

<img src='https://miro.medium.com/max/886/1*ZhGLUwaaqlJ9C0WK0nbAEA.png' width='300'>


**Custom learning rate**
$$\Large{lrate = d_{model}^{-0.5} * min(step{\_}num^{-0.5}, step{\_}num * warmup{\_}steps^{-1.5})}$$

Why **Smoothing** matters:

Without smoothing: The model is trained to be 100% confident, which leads to overconfident predictions, poor calibration, and overfitting.
With smoothing: The model learns to be slightly uncertain, which improves generalization, calibration, and often BLEU scores in translation tasks.

In [13]:
# AdamWarmup: implements the Noam learning rate schedule (warm-up + inverse sqrt decay)
class AdamWarmup:

    def __init__(self, model_size, warmup_steps, optimizer):
        self.model_size = model_size
        self.warmup_steps = warmup_steps
        self.optimizer = optimizer
        self.current_step = 0
        self.lr = 0

    # Compute current learning rate using the Noam formula
    def get_lr(self):
        return self.model_size ** (-0.5) * min(
            self.current_step ** (-0.5), self.current_step * self.warmup_steps ** (-1.5)
        )

    # Advance one step: update lr for all param groups and call optimizer.step()
    def step(self):
        # Increment the number of steps each time we call the step function
        self.current_step += 1
        lr = self.get_lr()
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr
        # update the learning rate
        self.lr = lr
        self.optimizer.step()


# LossWithLS: KL-divergence loss with label smoothing applied via a soft target distribution
class LossWithLS(nn.Module):

    def __init__(self, size, smooth):
        super(LossWithLS, self).__init__()
        self.criterion = nn.KLDivLoss(reduction="none")
        self.confidence = 1.0 - smooth
        self.smooth = smooth
        self.size = size

    def forward(self, prediction, target, mask):
        # Flatten prediction from (batch_num, output_len, vocab_size) → (batch_num * output_len, vocab_size)
        prediction = prediction.view(-1, prediction.size(-1))

        # Flatten target and mask
        # (batch_num, output_len) → (batch_num * output_len,)
        target = target.contiguous().view(-1)
        mask = mask.float().view(-1)

        # Build smooth label distribution: fill all entries with smooth / (vocab_size - 1)
        # Input/Output: (batch_num * output_len, vocab_size)
        labels = prediction.data.clone()
        labels.fill_(self.smooth / (self.size - 1))

        # Place high-confidence value (1 - smooth) at the ground-truth token index
        # target.unsqueeze(1): (batch_num * output_len, 1)
        labels.scatter_(1, target.data.unsqueeze(1), self.confidence)

        # Compute per-token KL-divergence loss then sum over vocab dim
        # (batch_num * output_len, vocab_size) → after sum(1): (batch_num * output_len,)
        loss = self.criterion(prediction, labels)

        # Apply mask and compute mean loss over non-padding tokens
        # Output: scalar
        loss = (loss.sum(1) * mask).sum() / mask.sum()
        return loss


# Instantiate transformer, optimizer with warmup schedule, and label-smoothed loss
transformer = Transformer(embed_dim=512, heads=8, num_layers=2, word_map=word_map)
transformer = transformer.to(DEVICE)
print(f"{sum(p.numel() for p in transformer.parameters()) / 1e6:.1f}M parameters")
adam_optimizer = torch.optim.Adam(
    transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9
)
# The paper warms up for 4000 of 100K steps; this demo trains ~1500 steps, so warm up for 500
transformer_optimizer = AdamWarmup(
    model_size=512, warmup_steps=500, optimizer=adam_optimizer
)
criterion = LossWithLS(len(word_map), 0.2)

# Quick forward pass test to verify output shape
# Input: question (batch_num, max_len), reply_input (batch_num, output_len + 1)
# Output: (batch_num, output_len + 1, vocab_size)
out = transformer(
    question.to(DEVICE),
    question_mask.to(DEVICE),
    reply_input.to(DEVICE),
    reply_input_mask.to(DEVICE),
)
print(out.size())

33.4M parameters


torch.Size([32, 26, 18243])


In [14]:
# Training loop: one epoch of mini-batch gradient descent over the DataLoader
def train(train_loader, transformer, criterion, epoch, device=DEVICE, max_steps=None):

    transformer.train()
    sum_loss = 0
    count = 0

    for i, (question, reply) in enumerate(train_loader):

        samples = question.shape[0]

        # Move tensors to target device
        # question: (batch_num, input_len), reply: (batch_num, output_len + 2)
        question = question.to(DEVICE)
        reply = reply.to(DEVICE)

        # Right-shift reply to produce teacher-forced input and target sequences
        # reply_input:  (batch_num, output_len + 1)
        # reply_target: (batch_num, output_len + 1)
        reply_input = reply[:, :-1]
        reply_target = reply[:, 1:]

        # Build all three masks
        # question_mask:     (batch_num, 1, 1, input_len)
        # reply_input_mask:  (batch_num, 1, output_len + 1, output_len + 1)
        # reply_target_mask: (batch_num, output_len + 1)
        question_mask, reply_input_mask, reply_target_mask = create_masks(
            question, reply_input, reply_target
        )

        # Forward pass through Transformer
        # Input: question (batch_num, input_len), reply_input (batch_num, output_len + 1)
        # Output: (batch_num, output_len + 1, vocab_size)
        out = transformer(question, question_mask, reply_input, reply_input_mask)

        # Compute label-smoothed KL-divergence loss
        # Input: out (batch_num, output_len + 1, vocab_size), reply_target (batch_num, output_len + 1)
        # Output: scalar
        loss = criterion(out, reply_target, reply_target_mask)

        # Backpropagation and parameter update with warmup scheduler
        transformer_optimizer.optimizer.zero_grad()
        loss.backward()
        transformer_optimizer.step()

        sum_loss += loss.item() * samples
        count += samples

        if i % 500 == 0:
            print(
                "Epoch [{}][{}/{}]	Loss: {:.3f}".format(
                    epoch, i, len(train_loader), sum_loss / count
                )
            )
        if max_steps is not None and i + 1 >= max_steps:
            break

In [15]:
# A short run (~2 min per 1,000 steps on an M-series GPU): loss falls from ~7.5 to ~4.4.
# The paper trained 100K steps on 8 GPUs. This little, greedy decoding collapses onto the most
# frequent reply in the corpus ("i dont know"), the classic failure of short seq2seq chat training;
# the model only starts conditioning on the question after many more steps.
NUM_STEPS = 1500
start = time.time()
train(train_loader, transformer, criterion, epoch=0, max_steps=NUM_STEPS)
print(f"{NUM_STEPS} steps in {time.time() - start:.0f}s")
if DEVICE.type == "mps":
    print(f"MPS memory allocated: {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
elif DEVICE.type == "cuda":
    print(f"Peak CUDA memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Epoch [0][0/6926]	Loss: 7.541


Epoch [0][500/6926]	Loss: 4.662


Epoch [0][1000/6926]	Loss: 4.481


1500 steps in 135s
MPS memory allocated: 0.90 GB


In [16]:
# Greedy decoding / evaluation: auto-regressively generates a reply one token at a time
def evaluate(transformer, question, question_mask, max_len, word_map, device=DEVICE):
    """
    Performs Greedy Decoding with a batch size of 1
    """
    # Build reverse mapping from integer index back to word token
    rev_word_map = {v: k for k, v in word_map.items()}
    # Set transformer to evaluation mode (disables dropout)
    transformer.eval()
    start_token = word_map["<start>"]

    # Encode the source question once
    # (1, input_len), mask (1, 1, 1, input_len) → (1, input_len, embed_dim)
    encoded = transformer.encode(question, question_mask)

    # Initialise decoder input with the <start> token
    # Output: (1, 1)
    words = torch.LongTensor([[start_token]]).to(DEVICE)

    # Auto-regressive decoding loop: generate one token per step
    for step in range(max_len - 1):
        size = words.shape[1]

        # Causal (look-ahead) mask for the currently generated sequence
        # Output: (1, 1, seq_len, seq_len)
        target_mask = torch.tril(
            torch.ones(size, size, dtype=torch.bool, device=DEVICE)
        )
        target_mask = target_mask.unsqueeze(0).unsqueeze(0)

        # Decode the sequence so far
        # Input: words (1, seq_len), encoded (1, input_len, embed_dim)
        # Output: (1, seq_len, embed_dim)
        decoded = transformer.decode(words, target_mask, encoded, question_mask)

        # Project only the last time-step to vocabulary logits
        # decoded[:, -1]: (1, embed_dim) → predictions: (1, vocab_size)
        predictions = transformer.logit(decoded[:, -1])

        # Greedy selection: pick the token with highest probability
        # (1, vocab_size) → next_word: scalar index
        next_word = predictions.argmax(dim=1).item()

        # Stop if the <end> token is generated
        if next_word == word_map["<end>"]:
            break

        # Append predicted token to the running sequence
        # (1, seq_len) cat (1, 1) → (1, seq_len + 1)
        words = torch.cat([words, torch.LongTensor([[next_word]]).to(DEVICE)], dim=1)

    # Strip <start> token and join remaining tokens into a sentence
    sen_idx = [w for w in words.squeeze(0).tolist() if w != start_token]
    return " ".join(rev_word_map[idx] for idx in sen_idx)


with torch.no_grad():
    for text in [
        "hi",
        "how are you",
        "what is your name",
        "where are you going",
        "do you love me",
    ]:
        # Tokenize, map unknown words to <unk>, add batch dimension
        # list of (num_tokens,) → (1, input_len)
        enc_qus = [
            word_map.get(word, word_map["<unk>"]) for word in remove_punc(text).split()
        ]
        question = torch.LongTensor(enc_qus).to(DEVICE).unsqueeze(0)

        # (1, input_len) → (1, 1, 1, input_len)
        question_mask = (question != 0).unsqueeze(1).unsqueeze(1)
        print(
            f"Q: {text}\nA: {evaluate(transformer, question, question_mask, 25, word_map)}\n"
        )

Q: hi
A: i dont know what i think i dont know



Q: how are you
A: i dont know what i think i dont know



Q: what is your name
A: i dont know what i think i dont know

Q: where are you going
A: i dont know what i think i dont know

Q: do you love me
A: i dont know what i think i dont know

